In [1]:
import json
from sentence_transformers import SentenceTransformer

c:\Users\sanoj\rbac_chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
with open("C:/Users/sanoj/rbac_chatbot/chunking_data.json" ,"r" ,encoding="utf-8") as file:
    chunks = json.load(file)


print(len(chunks))

222


In [5]:
texts = [c["text"] for c in chunks]
print(texts)

['## 1. Introduction\n\n### 1.1 Company Overview\nTechnologies is a leading abc company headquartered in Bangalore, India, with operations across North America, Europe, and Asia-Pacific. Founded in 2018, FinSolve provides innovative financial solutions, including digital banking, payment processing, wealth management, and enterprise financial analytics, serving over 2 million individual users and 10,000 businesses globally.', '### 1.2 Purpose\nThis engineering document outlines the technical architecture, development processes, and operational guidelines for FinSolve\'s product ecosystem. It serves as a comprehensive guide for engineering teams, stakeholders, and partners to ensure alignment with FinSolve\'s mission: "To empower financial freedom through secure, scalable, and innovative technology solutions."\n\n### 1.3 Scope\nThis document covers:', '### 1.3 Scope\nThis document covers:\n\n* System architecture and infrastructure\n* Software development lifecycle (SDLC)\n* Technology 

In [6]:
#load and create embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(texts, show_progress_bar=True)



c:\Users\sanoj\rbac_chatbot\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sanoj\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Batches: 100%|██████████| 7/7 [00:12<00:00,  1.74s/it]


In [9]:
embeddings = embeddings.tolist()

In [ ]:
with open("embeddings.json", "w", encoding="utf-8") as file:
    json.dump(embeddings, file, indent=2)
    print("embeddings.json successfully created")

embeddings.json successfully created


: 

In [5]:
###initialze pinecone
from pinecone import Pinecone, ServerlessSpec


index_name = "rbac-rag-chat"

In [ ]:
from pinecone import Pinecone
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("PINECONE_API_KEY")

print(api_key)

pc = Pinecone(api_key=api_key)

print(pc.list_indexes())

pcsk_7PDr7q_3py19hoSM2sZF8d7WNVnqDWztMPGqRZjCGmywbZoVtni4azQJQVNrNH7dYP9mRo
IndexList([<name='rbac-rag', dim=1024, ready=True>])


In [10]:


dimension = int(len(embeddings[0]))

existing_indexes = pc.list_indexes().names()

if index_name not in existing_indexes:

    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

print("Index ready")

Index ready


In [11]:
index = pc.Index(index_name)

In [13]:
###data upsertion

vectors = []

for i,(embedding,chunk) in enumerate(zip(embeddings,chunks)):
    vectors.append({
        "id": f"chunk-{i}",
        "values": embedding.tolist(),
        "metadata": {
            "text": chunk["text"],
            "department": chunk["department"],
            "file_name": chunk["file_name"],
            "chunk_id": chunk["chunk_id"]
        }
    })


print(vectors)

[{'id': 'chunk-0', 'values': [0.03596112132072449, -0.06234686076641083, -0.04356258362531662, -0.017998654395341873, -0.006242997944355011, -0.04454747959971428, 0.02328488416969776, -0.006651740986853838, 0.03514530509710312, -0.07653696089982986, -0.031697195023298264, 0.007484761066734791, 0.011810497380793095, 0.006465875077992678, -0.07190263271331787, -0.08373750001192093, -0.026362845674157143, -0.1422368586063385, 0.07087856531143188, -0.12265431880950928, -0.00812086183577776, -0.0696679875254631, 0.010060401633381844, -0.040983036160469055, -0.06475363671779633, 0.06474342197179794, -0.01362678688019514, -0.017791854217648506, 0.00279729044996202, -0.07786771655082703, -0.04080571234226227, 0.05633566528558731, 0.035749394446611404, -0.006971652619540691, 0.0022357089910656214, -0.014222738333046436, 0.03984227403998375, -0.005834463983774185, -0.06021423265337944, 0.035898350179195404, -0.056188639253377914, -0.038787055760622025, 0.022189820185303688, 0.06909313052892685, 

In [14]:
index.upsert(vectors=vectors)

print("Uploaded to Pinecone successfully")

Uploaded to Pinecone successfully
